# Module 06: Inspecting MRIQC Reports and Making Exclusion Decisions

In this module we review image quality metrics (IQMs) produced by MRIQC, set exclusion thresholds, and document which participants will be excluded from further analysis.

## Why QC Decisions Matter for Replication

Transparent, pre-registered QC decisions are essential for reproducibility. Arbitrary or post-hoc exclusions inflate false-positive rates. By documenting every threshold and rationale before unblinding your results, you ensure that other researchers can replicate your sample exactly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Key Metrics for T1w Exclusion

| Metric | Description | Typical good range |
|--------|-------------|--------------------|
| `cnr`  | Contrast-to-noise ratio | > 2.0 |
| `snr`  | Signal-to-noise ratio | > 10 |
| `cjv`  | Coefficient of joint variation (lower = better) | < 0.6 |
| `fwhm_avg` | Average smoothness estimate | < 3.0 |

In [ ]:
n = 20
t1w_iqms = pd.DataFrame({
    'subject_id': [f'sub-{i:02d}' for i in range(1, n + 1)],
    'cnr':       np.random.normal(2.5, 0.4, n),
    'snr':       np.random.normal(12.0, 2.0, n),
    'cjv':       np.random.normal(0.55, 0.1, n),
    'fwhm_avg':  np.random.normal(2.5, 0.3, n),
})
# Inject a couple of outliers
t1w_iqms.loc[3, 'cjv'] = 0.85
t1w_iqms.loc[11, 'snr'] = 5.0

t1w_iqms['t1w_outlier'] = (t1w_iqms['cjv'] > 0.7) | (t1w_iqms['snr'] < 8)
print(t1w_iqms.to_string(index=False))

## 2. Key Metrics for BOLD Exclusion

| Metric | Description | Typical good range |
|--------|-------------|--------------------|
| `tsnr` | Temporal SNR | > 40 |
| `mean_fd` | Mean framewise displacement (mm) | < 0.5 |
| `dvars_nstd` | Normalised DVARS | < 1.5 |
| `aor` | AFNI outlier ratio | < 0.2 |
| `gcor` | Global correlation | < 0.5 |

In [ ]:
bold_iqms = pd.DataFrame({
    'subject_id':  [f'sub-{i:02d}' for i in range(1, n + 1)],
    'tsnr':        np.random.normal(55, 10, n),
    'mean_fd':     np.abs(np.random.normal(0.25, 0.15, n)),
    'dvars_nstd':  np.random.normal(1.2, 0.2, n),
    'aor':         np.random.normal(0.1, 0.05, n),
    'gcor':        np.random.normal(0.3, 0.08, n),
})
# Inject outliers
bold_iqms.loc[5, 'mean_fd'] = 0.75
bold_iqms.loc[14, 'tsnr'] = 28.0

bold_iqms['bold_outlier'] = (bold_iqms['mean_fd'] > 0.5) | (bold_iqms['tsnr'] < 40)
print(bold_iqms.to_string(index=False))

## 3. Setting Exclusion Thresholds

We pre-register the following thresholds:

- **mean_fd > 0.5 mm** → exclude (excessive motion)
- **tSNR < 40** → exclude (poor signal quality)
- **cjv > 0.7** → exclude (poor T1w quality)

In [ ]:
FD_THRESHOLD   = 0.5
TSNR_THRESHOLD = 40.0
CJV_THRESHOLD  = 0.7

bold_iqms['exclude'] = (bold_iqms['mean_fd'] > FD_THRESHOLD) | (bold_iqms['tsnr'] < TSNR_THRESHOLD)
t1w_iqms['exclude']  = t1w_iqms['cjv'] > CJV_THRESHOLD

print(f"BOLD exclusions : {bold_iqms['exclude'].sum()} / {len(bold_iqms)}")
print(f"T1w  exclusions : {t1w_iqms['exclude'].sum()} / {len(t1w_iqms)}")

## 4. Visualizing QC Metrics

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=bold_iqms, x='mean_fd', y='tsnr',
    hue='exclude', palette={True: 'red', False: 'steelblue'},
    s=80, ax=ax
)
ax.axvline(FD_THRESHOLD, ls='--', color='gray', label=f'FD = {FD_THRESHOLD}')
ax.axhline(TSNR_THRESHOLD, ls=':', color='gray', label=f'tSNR = {TSNR_THRESHOLD}')
ax.set_xlabel('Mean Framewise Displacement (mm)')
ax.set_ylabel('Temporal SNR')
ax.set_title('BOLD QC: tSNR vs Mean FD')
ax.legend(title='Excluded')
plt.tight_layout()
plt.show()

## 5. Documenting Exclusion Decisions

We save a `participants_qc.tsv` that merges both sets of IQMs and records which participants are excluded and why.

In [ ]:
merged = bold_iqms[['subject_id', 'tsnr', 'mean_fd', 'exclude']].rename(
    columns={'exclude': 'bold_exclude'}
).merge(
    t1w_iqms[['subject_id', 'cnr', 'snr', 'cjv', 'exclude']].rename(
        columns={'exclude': 't1w_exclude'}
    ),
    on='subject_id'
)
merged['exclude'] = merged['bold_exclude'] | merged['t1w_exclude']

output_path = 'participants_qc.tsv'
merged.to_csv(output_path, sep='\t', index=False)
print(f'Saved {output_path}')
print(merged[merged['exclude']].to_string(index=False))

## Summary

In this module you:
- Reviewed key IQMs for both T1w and BOLD data
- Applied pre-registered exclusion thresholds
- Visualised motion and signal-quality trade-offs
- Documented exclusion decisions in `participants_qc.tsv`

**Next steps → Module 07: Preprocessing with fMRIPrep**